# MiniAn Pipeline Separation Judgement

This notebook loads the metrics and pass/fail results from
`separation/tools/run_separation_judgement.py` and produces
visualizations comparing the monolithic and separated pipelines.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

## 1. Load Results

In [ ]:
output_root = Path('output/separation_judgement')

metrics_csv = output_root / 'metrics.csv'
pass_fail_csv = output_root / 'pass_fail.csv'
timings_json = output_root / 'timings.json'

assert metrics_csv.exists(), f'Run separation/tools/run_separation_judgement.py first. Missing: {metrics_csv}'

df_metrics = pd.read_csv(metrics_csv)
df_pass_fail = pd.read_csv(pass_fail_csv)

with open(timings_json) as f:
    timings = json.load(f)

print(f'Metrics rows: {len(df_metrics)}')
print(f'Pass/Fail rows: {len(df_pass_fail)}')
print(f'Timings: {timings}')

## 2. Pass/Fail Summary

In [ ]:
# Display pass/fail table with color coding
def highlight_pass_fail(val):
    if val == True or val == 'True':
        return 'background-color: #90EE90'
    elif val == False or val == 'False':
        return 'background-color: #FFB6C1'
    return ''

styled = df_pass_fail.style.applymap(highlight_pass_fail, subset=['pass'])
display(styled)

all_pass = df_pass_fail['pass'].all()
print(f'\nOverall verdict: {"PASS" if all_pass else "FAIL"}')

## 3. Detailed Gate Metrics

In [ ]:
# Show all gate metrics
gate_df = df_metrics[df_metrics['is_gate'] == True].copy()
display(gate_df[['module', 'target', 'exact_equal', 'max_abs_error', 'pearson_corr', 'note']])

## 4. Spatial Footprint Comparison

In [ ]:
dataset = 'demo_movies'
orig_dir = output_root / 'original' / dataset
sep_dir = output_root / 'separated' / dataset

if orig_dir.exists() and sep_dir.exists():
    A_orig = np.load(orig_dir / 'A.npy')
    A_sep = np.load(sep_dir / 'A.npy')
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Max projection of spatial footprints
    proj_orig = np.max(A_orig, axis=0) if A_orig.ndim == 3 else A_orig
    proj_sep = np.max(A_sep, axis=0) if A_sep.ndim == 3 else A_sep
    
    axes[0].imshow(proj_orig, cmap='viridis')
    axes[0].set_title(f'Original (n={A_orig.shape[0]})')
    axes[0].axis('off')
    
    axes[1].imshow(proj_sep, cmap='viridis')
    axes[1].set_title(f'Separated (n={A_sep.shape[0]})')
    axes[1].axis('off')
    
    # Difference map
    if proj_orig.shape == proj_sep.shape:
        diff = proj_orig - proj_sep
        im = axes[2].imshow(diff, cmap='RdBu_r', vmin=-np.max(np.abs(diff)), vmax=np.max(np.abs(diff)))
        axes[2].set_title(f'Difference (max={np.max(np.abs(diff)):.6f})')
        plt.colorbar(im, ax=axes[2])
    else:
        axes[2].text(0.5, 0.5, 'Shape mismatch', ha='center', va='center', transform=axes[2].transAxes)
    axes[2].axis('off')
    
    plt.suptitle('Spatial Footprint Comparison (A max projection)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Output directories not found. Run the judgement script first.')

## 5. Temporal Trace Comparison

In [ ]:
if orig_dir.exists() and sep_dir.exists():
    C_orig = np.load(orig_dir / 'C.npy')
    C_sep = np.load(sep_dir / 'C.npy')
    
    n_show = min(5, C_orig.shape[0], C_sep.shape[0])
    
    fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show), sharex=True)
    if n_show == 1:
        axes = [axes]
    
    for i in range(n_show):
        axes[i].plot(C_orig[i], label='Original', alpha=0.8)
        axes[i].plot(C_sep[i], label='Separated', alpha=0.8, linestyle='--')
        axes[i].set_ylabel(f'Unit {i}')
        if i == 0:
            axes[i].legend()
    
    axes[-1].set_xlabel('Frame')
    plt.suptitle('Temporal Trace Comparison (C)', fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print('Output directories not found.')

## 6. Motion Correction Comparison

In [ ]:
if orig_dir.exists() and sep_dir.exists():
    motion_orig = np.load(orig_dir / 'motion.npy')
    motion_sep = np.load(sep_dir / 'motion.npy')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    for dim in range(motion_orig.shape[1]):
        label = ['height', 'width'][dim]
        axes[0].plot(motion_orig[:, dim], label=f'Orig {label}')
        axes[0].plot(motion_sep[:, dim], '--', label=f'Sep {label}')
    axes[0].set_title('Motion Shifts')
    axes[0].set_xlabel('Frame')
    axes[0].set_ylabel('Shift (pixels)')
    axes[0].legend()
    
    diff = motion_orig - motion_sep
    axes[1].plot(diff)
    axes[1].set_title(f'Motion Difference (max abs={np.max(np.abs(diff)):.6f})')
    axes[1].set_xlabel('Frame')
    axes[1].set_ylabel('Difference')
    
    plt.tight_layout()
    plt.show()
else:
    print('Output directories not found.')

## 7. Correlation and SSIM Statistics

In [ ]:
# Summary statistics
print('=== Correlation Statistics ===')
corr_vals = df_metrics['pearson_corr'].dropna()
if len(corr_vals) > 0:
    print(f'  Mean: {corr_vals.mean():.6f}')
    print(f'  Min:  {corr_vals.min():.6f}')
    print(f'  Max:  {corr_vals.max():.6f}')

print('\n=== SSIM Statistics ===')
ssim_vals = df_metrics['ssim'].dropna()
if len(ssim_vals) > 0:
    print(f'  Mean: {ssim_vals.mean():.6f}')
    print(f'  Min:  {ssim_vals.min():.6f}')
    print(f'  Max:  {ssim_vals.max():.6f}')

print('\n=== Max Absolute Error ===')
mae_vals = df_metrics['max_abs_error'].dropna()
if len(mae_vals) > 0:
    print(f'  Mean: {mae_vals.mean():.6f}')
    print(f'  Max:  {mae_vals.max():.6f}')

## 8. Final Verdict

In [ ]:
gate_results = df_metrics[df_metrics['is_gate'] == True]
n_gates = len(gate_results)
n_pass = gate_results['exact_equal'].sum()
n_fail = n_gates - n_pass

print(f'Gate metrics: {n_pass}/{n_gates} PASS')
if n_fail > 0:
    print(f'\nFailed gates:')
    failed = gate_results[gate_results['exact_equal'] == False]
    for _, row in failed.iterrows():
        print(f'  {row["module"]}/{row["target"]}: {row["note"]}')
    print(f'\nVERDICT: FAIL')
else:
    print(f'\nVERDICT: PASS — Separated pipeline produces identical results to monolithic.')
    print(f'\nTiming: original={timings.get("original_sec", "?"):.1f}s, '
          f'separated={timings.get("separated_sec", "?"):.1f}s')